# Main empirical study: Lalonde job-training data

The primary estimand is the standard ATT for NSW treated units. ATE is reported as a sensitivity analysis. The notebook uses the Lalonde data file stored under `notebooks/experiments/data`. Missing files stop execution.


In [ ]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise RuntimeError("Run this notebook from inside the genriesz repository.")
    REPO_ROOT = parent

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from genriesz import grr_ate, grr_att
from genriesz.basis import BaseBasis, TreatmentInteractionBasis
from genriesz.experiments import (
    COMPATIBLE_LOSSES,
    ESTIMANDS,
    ESTIMATORS_ALL,
    RANDOM_SEED,
    TREATMENT_INDEX,
    CoverageDiagnosticBasis,
    SelectedColumnsBasis,
    fit_grr_love_plot_data,
    fit_matching_ate,
    fit_one_grr,
    fit_one_grr_with_basis,
    fit_one_incompatible,
    fit_one_plugin_logistic,
    generator_shift_for_estimand,
    load_ihdp_replication,
    load_lalonde,
    make_basis,
    make_compatible_generator,
    make_coverage_diagnostic_data,
    make_dimension_data,
    make_kang_schafer_data,
    make_kernel_gp_data,
    make_score_guided_data,
    make_simulation_data,
    result_to_rows,
    summarize_estimates,
    true_theta,
)

DATA_DIR = REPO_ROOT / "notebooks" / "experiments" / "data"
TABLE_CONFIG = {"float_format": "{:.4f}", "max_rows": 200}
PLOT_CONFIG = {
    "figure_size": (9.0, 5.2),
    "figure_size_wide": (12.0, 5.2),
    "title_fontsize": 14,
    "axis_fontsize": 12,
    "tick_fontsize": 10,
    "legend_fontsize": 10,
    "line_width": 2.0,
    "marker_size": 5,
    "box_width": 0.70,
    "grid_alpha": 0.30,
    "dpi": 140,
    "squared_error_y_scale": "log",
    "squared_error_floor": 1e-12,
}
METHOD_LABELS = {
    "ra": "RA",
    "rw": "RW (IPW)",
    "arw": "ARW (AIPW)",
    "tmle": "TMLE",
    "SQ": "SQ-Riesz",
    "UKL": "UKL-Riesz",
    "BKL": "BKL-Riesz",
    "BP(0.5)": "BP-Riesz (omega = 0.5)",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random forest leaves",
    "rff": "Random Fourier features",
    "matching": "Nearest-neighbor matching",
}
METHOD_COLORS = {
    "SQ": "#4C78A8",
    "UKL": "#F58518",
    "BKL": "#54A24B",
    "BP(0.5)": "#B279A2",
    "rkhs": "#4C78A8",
    "polynomial": "#F58518",
    "rf": "#54A24B",
    "rff": "#E45756",
    "matching": "#72B7B2",
}
DISPLAY_LABELS = {
    "ra": "RA",
    "rw": "RW",
    "arw": "ARW",
    "tmle": "TMLE",
    "rkhs": "RKHS",
    "polynomial": "Polynomial",
    "rf": "Random Forest",
    "rff": "Random Fourier Features",
    "matching": "Nearest-Neighbor Matching",
    "regressor": "Regressor",
    "covariate": "Covariate",
}
LABEL_COLUMNS = ("estimator", "basis", "basis_mode", "loss", "loss_link_pair")
pd.options.display.max_rows = TABLE_CONFIG["max_rows"]


def label_of(value):
    """Return the display label for a stored result key."""

    return DISPLAY_LABELS.get(str(value), str(value))


def prettify_method(text):
    """Replace stored result keys inside a composite display label."""

    out = str(text)
    for key, value in DISPLAY_LABELS.items():
        out = re.sub(r"(?<![A-Za-z0-9_])" + re.escape(key) + r"(?![A-Za-z0-9_])", value, out)
    return out


def prettify_labels(frame):
    """Return a copy with known result-key columns formatted for display."""

    out = frame.copy()
    for column in LABEL_COLUMNS:
        if column in out.columns:
            out[column] = out[column].map(label_of)
    return out


def display_table(frame, *, caption=None, digits=4):
    """Display a rounded table without changing the stored results."""

    table = prettify_labels(frame)
    numeric_columns = table.select_dtypes(include=[np.number]).columns
    table[numeric_columns] = table[numeric_columns].round(digits)
    styler = table.style.format(precision=digits)
    if caption is not None:
        styler = styler.set_caption(caption)
    display(styler)


In [ ]:
BASIS_KIND = "rkhs"
RIEZ_FEATURES = 100
FOLDS = 5
RIEZ_LAMBDA = 3e-1
PRIMARY_ESTIMAND = "ATT"
SENSITIVITY_ESTIMAND = "ATE"
print({"primary": PRIMARY_ESTIMAND, "sensitivity": SENSITIVITY_ESTIMAND})

In [ ]:
data = load_lalonde()
print("Data source:", data["source"])
print("n =", len(data["Y"]), "treated =", int(data["D"].sum()), "controls =", int((1 - data["D"]).sum()))
rows = []
for loss_spec in COMPATIBLE_LOSSES:
    for estimand in [PRIMARY_ESTIMAND, SENSITIVITY_ESTIMAND]:
        fit_rows = fit_one_grr(
            data,
            estimand=estimand,
            loss_spec=loss_spec,
            basis_kind=BASIS_KIND,
            basis_mode="regressor",
            cross_fit=True,
            lam=RIEZ_LAMBDA,
            basis_features=RIEZ_FEATURES,
            folds=FOLDS,
            estimators=ESTIMATORS_ALL,
            random_state=123,
        )
        rows.extend(fit_rows)
lalonde_results = pd.DataFrame(rows)
summary_cols = ["estimand", "loss", "estimator", "estimate", "se", "ci_low", "ci_high", "p_value", "alpha_abs_p95", "alpha_abs_max", "max_abs_smd_weighted", "ess_treated", "ess_control", "riesz_clip_binding_rate_max", "riesz_modifies_estimand", "status"]
summary_cols = [c for c in summary_cols if c in lalonde_results.columns]

for estimand_name, caption_suffix in [(PRIMARY_ESTIMAND, "primary ATT for NSW treated"), (SENSITIVITY_ESTIMAND, "ATE sensitivity analysis")]:
    table_df = lalonde_results[lalonde_results["estimand"] == estimand_name][summary_cols].copy()
    if table_df.empty:
        print(f"No rows for {estimand_name}.")
        continue
    table_df = table_df.drop(columns=["estimand"])
    table_df = table_df.sort_values(["loss", "estimator"])
    display_table(table_df, caption=f"Lalonde {caption_suffix}")


In [ ]:
# Forest plots with 95% confidence intervals. ATT and ATE are displayed separately.
plot_df = lalonde_results[(lalonde_results["status"] == "ok") & (lalonde_results["estimator"].isin(["rw", "arw", "tmle"]))].copy()
plot_df["method"] = plot_df["loss"] + " | " + plot_df["estimator"]

for estimand_name, caption_suffix in [(PRIMARY_ESTIMAND, "primary ATT"), (SENSITIVITY_ESTIMAND, "ATE sensitivity")]:
    panel_df = plot_df[plot_df["estimand"] == estimand_name].copy()
    if panel_df.empty:
        print(f"No plot data for {estimand_name}.")
        continue
    panel_df = panel_df.sort_values(["loss", "estimator"]).reset_index(drop=True)
    y = np.arange(len(panel_df))
    fig, ax = plt.subplots(figsize=(8.0, max(4.0, 0.35 * len(panel_df))), dpi=PLOT_CONFIG["dpi"])
    xerr = np.vstack([panel_df["estimate"] - panel_df["ci_low"], panel_df["ci_high"] - panel_df["estimate"]])
    ax.errorbar(panel_df["estimate"], y, xerr=xerr, fmt="o", color="#4C78A8", ecolor="#4C78A8", linewidth=1.2, capsize=3)
    ax.set_yticks(y)
    ax.set_yticklabels([prettify_method(_m) for _m in panel_df["method"]], fontsize=PLOT_CONFIG["tick_fontsize"])
    ax.invert_yaxis()
    ax.set_xlabel("Estimate", fontsize=PLOT_CONFIG["axis_fontsize"])
    ax.set_title(f"Lalonde {caption_suffix}: estimates with 95% confidence intervals", fontsize=PLOT_CONFIG["title_fontsize"])
    ax.axvline(0.0, color="black", linewidth=1.0, linestyle=":")
    ax.grid(axis="x", alpha=PLOT_CONFIG["grid_alpha"])
    fig.tight_layout()
    plt.show()


In [ ]:
# Love plot for a representative primary ATT fit. Change SELECTED_LOSS to edit the plotted method.
SELECTED_LOSS = "UKL"
selected_spec = next(s for s in COMPATIBLE_LOSSES if s["label"] == SELECTED_LOSS)
selected_basis = make_basis(BASIS_KIND, mode="regressor", seed=123, n_features=RIEZ_FEATURES)
selected_gen = make_compatible_generator(selected_spec["loss"], estimand="ATT", omega=selected_spec.get("omega"))
selected_res = grr_att(X=data["X"], Y=data["Y"], basis=selected_basis, generator=selected_gen, treatment_index=TREATMENT_INDEX, cross_fit=True, folds=FOLDS, riesz_lam=RIEZ_LAMBDA, estimators=ESTIMATORS_ALL, max_iter=500)
love = selected_res.love_plot_data(as_pandas=True)
love = love.sort_values("abs_smd_weighted", ascending=False).head(25)
fig, ax = plt.subplots(figsize=(8.0, max(4.0, 0.25 * len(love))), dpi=PLOT_CONFIG["dpi"])
y = np.arange(len(love))
ax.scatter(love["abs_smd_unweighted"], y, label="Unweighted", color="#999999", s=30)
ax.scatter(love["abs_smd_weighted"], y, label="Weighted", color=METHOD_COLORS.get(SELECTED_LOSS, "#4C78A8"), s=30)
ax.axvline(0.10, color="black", linestyle="--", linewidth=1.0)
ax.set_yticks(y)
ax.set_yticklabels(love["covariate"], fontsize=PLOT_CONFIG["tick_fontsize"])
ax.invert_yaxis()
ax.set_xlabel("Absolute standardized mean difference", fontsize=PLOT_CONFIG["axis_fontsize"])
ax.set_title(f"Lalonde ATT Love plot: {SELECTED_LOSS}", fontsize=PLOT_CONFIG["title_fontsize"])
ax.legend(fontsize=PLOT_CONFIG["legend_fontsize"])
ax.grid(axis="x", alpha=PLOT_CONFIG["grid_alpha"])
fig.tight_layout()
plt.show()